# Notebook 2: Your First Neural Network
**Estimated time:** 30-45 min
**Prerequisites:** Notebook 1 (tensors and autograd)

---
## What you will learn
1. What `nn.Module` is and why we use it
2. How `nn.Linear` works mathematically
3. How to build and inspect a neural network
4. What a forward pass does
5. How activation functions add power to linear layers

## 1. What is a Neural Network?

**Imagine a series of workers in a factory assembly line.**

- Each **worker** (layer) takes in some information, does a simple job, and passes the result to the next worker.
- The **first worker** sees the raw input (e.g., pixel values of an image).
- The **last worker** produces the final answer (e.g., "this is a cat").
- Each worker has **knobs** (weights and biases) that can be tuned.
- **Training** = adjusting those knobs so the factory produces correct outputs.

A **linear layer** is the core worker. It does:
```
output = input @ weight.T + bias
```

That's literally matrix multiplication plus a vector addition.
But stacking many linear layers without anything in between is useless — they collapse into a single linear transformation.

**Activation functions** add non-linearity, letting the network learn curves, not just straight lines.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Let's see what nn.Linear does
layer = nn.Linear(in_features=3, out_features=2)

print('weight shape:', layer.weight.shape)   # (out, in) = (2, 3)
print('bias shape  :', layer.bias.shape)     # (out,) = (2,)
print()
print('weight:', layer.weight)
print('bias  :', layer.bias)

# Forward pass on one sample
x = torch.tensor([1.0, 2.0, 3.0])
out = layer(x)
print()
print('input  shape:', x.shape)
print('output shape:', out.shape)
print('output:', out)

# Verify manually:
manual = x @ layer.weight.T + layer.bias
print('manual (same):', manual)

## 2. Building a Network with `nn.Module`

`nn.Module` is PyTorch's base class for ALL neural networks.
Every network you build should inherit from it.

When you subclass `nn.Module` you need to:
1. Call `super().__init__()` in `__init__`
2. Define your layers in `__init__`
3. Implement `forward(self, x)` — describes how data flows through the network

In [ ]:
class MyFirstNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        # Define the layers (these are registered as parameters automatically)
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Describe data flow: input → layer1 → relu → layer2 → output
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


# Create a network: 4 inputs → 8 hidden → 2 outputs
model = MyFirstNetwork(input_size=4, hidden_size=8, output_size=2)
print(model)

In [ ]:
# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total}')
print(f'Trainable parameters: {trainable}')

# Run a forward pass
x = torch.randn(1, 4)    # batch of 1 sample, 4 features
out = model(x)
print(f'Input shape : {x.shape}')
print(f'Output shape: {out.shape}')

## 3. Batched Inputs

In practice we **never** feed one sample at a time. We use **batches**.

The first dimension is always the **batch dimension**:
- Input shape: `(batch_size, features)`
- Output shape: `(batch_size, outputs)`

PyTorch linear layers handle this automatically.

In [ ]:
# Batch of 32 samples, each with 4 features
batch = torch.randn(32, 4)
out = model(batch)
print(f'Batch input  shape: {batch.shape}')
print(f'Batch output shape: {out.shape}')

## 4. Activation Functions

Without activations, a stack of linear layers is just one linear layer.
Activations let networks learn non-linear patterns.

In [ ]:
x = torch.linspace(-5, 5, 100)

activations = {
    'ReLU'     : nn.ReLU(),
    'LeakyReLU': nn.LeakyReLU(0.1),
    'Sigmoid'  : nn.Sigmoid(),
    'Tanh'     : nn.Tanh(),
    'GELU'     : nn.GELU(),
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, (name, fn) in zip(axes, activations.items()):
    with torch.no_grad():
        y = fn(x)
    ax.plot(x.numpy(), y.numpy())
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name)
    ax.set_ylim(-1.5, 5)
plt.tight_layout()
plt.show()

print('''
ReLU      - Simple. Fast. Most common hidden layer activation.
LeakyReLU - Like ReLU but doesn't die for negative inputs.
Sigmoid   - Squashes to (0,1). Used in binary output layers.
Tanh      - Squashes to (-1,1). Good for hidden layers in RNNs.
GELU      - Smooth ReLU. Used in transformers (BERT, GPT).
''')

## 5. `nn.Sequential` — Quick Network Builder

For simple feedforward networks, you can skip writing a class and use `nn.Sequential`.

In [ ]:
# Equivalent to MyFirstNetwork above, but without writing a class
model2 = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

print(model2)
x = torch.randn(32, 4)
out = model2(x)
print('output shape:', out.shape)

## 6. Inspecting Weights and Setting Them

Understanding how to access and modify weights is useful for debugging and custom initialization.

In [ ]:
# Access weights directly
for name, param in model.named_parameters():
    print(f'{name:20s}  shape={str(param.shape):15s}  mean={param.data.mean():.4f}')

In [ ]:
# PyTorch default init is often good, but you can customize
import torch.nn.init as init

def init_weights(m):
    if isinstance(m, nn.Linear):
        init.xavier_uniform_(m.weight)   # good for tanh
        init.zeros_(m.bias)

model.apply(init_weights)
print('Weights re-initialized')

---
## YOUR TURN — Mini Project: The XOR Problem

XOR is a classic challenge that **cannot** be solved by a single linear layer, but **can** be solved by a 2-layer network.

XOR truth table:
```
Input       Output
[0, 0]  →     0
[0, 1]  →     1
[1, 0]  →     1
[1, 1]  →     0
```

**Task:**
1. Define the XOR dataset (4 samples, inputs and targets shown above)
2. Build a small network: `nn.Linear(2, 4)` → `nn.ReLU()` → `nn.Linear(4, 1)` → `nn.Sigmoid()`
3. Use `nn.BCELoss()` as loss and `torch.optim.Adam(model.parameters(), lr=0.1)` as optimizer
4. Write the training loop manually (we cover it properly in Notebook 3):
   - `optimizer.zero_grad()` → forward → loss → `loss.backward()` → `optimizer.step()`
5. Train for 1000 epochs, print loss every 200 epochs
6. After training, print predictions on all 4 inputs — they should be close to 0 or 1

**Why can't a single linear layer solve XOR?**
A linear layer can only draw a straight line (or hyperplane). XOR is not linearly separable — no straight line can separate the 0s from the 1s. You need a curve, which requires non-linearity.

In [ ]:
# YOUR CODE HERE

# 1. Define the dataset
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = torch.tensor([[0.0],      [1.0],       [1.0],       [0.0]])

# 2. Build the model

# 3. Define loss and optimizer

# 4. Training loop (1000 epochs)

# 5. After training, print predictions